# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced via their Croissant schema `@id` for clarity and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema (JSON-LD) available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the Croissant metadata and record sets using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic dataset info (access object attributes, not dict subscripting)
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Cite as: {dataset.metadata.cite_as}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Inspect the available record sets, their fields, and all `@id`s.

All further dataset elements will be referenced by their `@id` (as defined in the schema).

In [ ]:
# List all record sets and their field @ids
print("Available record sets in the dataset schema:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Fields (@id):")
    for field in record_set.fields:
        print(f"     - {field.id}  (name: {field.name}, type: {field.data_type})")
    print()

## 3. Data Extraction
Extract records from a specific record set. Fields and record sets are identified by their `@id` per the Croissant schema.

**Below, we select the primary protein quantification data record set for analysis.**

In [ ]:
# For this dataset, find the main quantification table record set (inspect the printout from the previous cell).
# Example: Let's suppose the quantification table record set has @id 'cr:QuantificationTable'.
# You can view all record_set.id by running the previous cell.
# Replace with the actual @id from the schema if it differs.
main_record_set_id = 'cr:QuantificationTable'  # <-- Confirm this is correct by inspecting available record_sets
record_sets_of_interest = [main_record_set_id]
dfs = {}

for rec_set_id in record_sets_of_interest:
    # Retrieve records using mlcroissant
    recs = list(dataset.records(record_set=rec_set_id))
    dfs[rec_set_id] = pd.DataFrame(recs)

df = dfs[main_record_set_id]
print(f"Columns in `{main_record_set_id}`: \n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic filtering, normalization, and grouping using fields referenced by their `@id`.

*We'll use the numeric field `cr:Abundance` (replace the `@id` with the actual numeric field from your printed columns).*

*Group by `cr:Sample` (again, ensure the reference matches a field from the field list printout).*

These IDs can be confirmed from the metadata inspection above.

In [ ]:
# Use the ID of a numeric field (e.g., cr:Abundance) and a grouping field (e.g., cr:Sample) from the dataset
numeric_field_id = 'cr:Abundance'    # Replace with actual numeric field @id from the schema
group_field_id = 'cr:Sample'        # Replace with actual group field @id from the schema
threshold = 10

# Filter for abundance > threshold
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize Abundance
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by sample
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean of numeric fields):")
        display(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame. Please check available columns and field @id.")

## 5. Visualization
Let's visualize the abundance distribution and abundance per sample using the field `cr:Abundance` (replace with the actual field `@id` if required).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of abundance
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} across all proteins")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot abundance per sample if grouping field available
    if group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"Cannot plot: {numeric_field_id} not found in columns. Columns are: {df.columns.tolist()}")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze a Croissant-described dataset using the `mlcroissant` library. All entities were referenced by their `@id` to ensure schema-awareness and reproducibility. The FAIR^2 dataset provides rich protein abundance data suitable for advanced downstream proteomics analysis.

**To examine or export other record sets, repeat the extraction and EDA steps using their `@id` as shown above.**